In [13]:
!pip install -q transformers==4.46.3 sentencepiece protobuf

import glob, json, os, time, torch
from kaggle_secrets import UserSecretsClient
!pip install -q transformers==4.46.3 sentencepiece protobuf

import glob, json, os, time, torch
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, T5Tokenizer

tok_hf = UserSecretsClient().get_secret("HF_TOKEN")

D = os.path.dirname(glob.glob("/kaggle/input/**/train.py", recursive=True)[0])
src = [json.loads(l)["source"] for l in open(f"{D}/test_samer_S0.jsonl", encoding="utf-8")]

def bench(repo, prefix, slow_t5):
    tok = (T5Tokenizer if slow_t5 else AutoTokenizer).from_pretrained(repo, token=tok_hf)
    model = AutoModelForSeq2SeqLM.from_pretrained(repo, token=tok_hf, torch_dtype=torch.float16).cuda().eval()
    order = sorted(range(len(src)), key=lambda i: len(src[i]))
    
    torch.cuda.synchronize()
    t = time.time()
!pip install -q transformers==4.46.3 sentencepiece protobuf

import glob, json, os, time, torch
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tok_hf = UserSecretsClient().get_secret("HF_TOKEN")

D = os.path.dirname(glob.glob("/kaggle/input/**/train.py", recursive=True)[0])
src = [json.loads(l)["source"] for l in open(f"{D}/test_samer_S0.jsonl", encoding="utf-8")]

def bench(repo, prefix):
    tok = AutoTokenizer.from_pretrained(repo, token=tok_hf)
    model = AutoModelForSeq2SeqLM.from_pretrained(repo, token=tok_hf, torch_dtype=torch.float16).cuda().eval()
    order = sorted(range(len(src)), key=lambda i: len(src[i]))
    
    torch.cuda.synchronize()
    t = time.time()
    
    with torch.no_grad():
        for k in range(0, len(order), 64):
            enc = tok([prefix + src[i] for i in order[k:k + 64]], padding=True, truncation=True,
                      max_length=256, return_tensors="pt").to("cuda")
            model.generate(**enc, max_length=256, num_beams=1, do_sample=False, no_repeat_ngram_size=4)
            
    torch.cuda.synchronize()
    s = time.time() - t
    print(f"{repo}: {len(src)} sentences in {s:.0f}s = {len(src) / s:.1f} sentences/s")

# 1. Benchmark AraT5
bench("Congi-libya/samer-arat5v2-base-simplification", "بسّط: ")

# 2. Benchmark AraBART
bench("Congi-libya/bayan-model2-arabart", "بسط: ")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Congi-libya/samer-arat5v2-base-simplification: 3277 sentences in 41s = 80.8 sentences/s
Congi-libya/bayan-model2-arabart: 3277 sentences in 11s = 292.8 sentences/s
